## Extracting high level dataset from the JSON captures

This dataset contains an extensive amount of features, intended to be filtered, depending on which method we use for the data synthesis. The features used for training the CTGAN for the hybrid approach will be very different than the ones used for the baseline model, as the subsequent low-level data generation tasks require diverse features as conditions. 

For the baseline model the features planned to use can e found in the baseline_simple_mode.md file. For the hybrid approach, we still have to figure out the best features, but generally it can be seen in the CTGAN notebooks what we have tried

The exported .json file from Wireshark a lot of times contains duplicate keys in an object, e.g. when there are multiple quic packets in one UDP packet.

In the .json we can see something like:

    "quic": {
    ...
    "quic.frame": { "quic.frame_type": "0x02", ... },
    "quic.frame": { "quic.frame_type": "0x18", ... },
    "quic.frame": { "quic.frame_type": "0x1e", ... },
    ...
    }

For this reason we need a custom JSON parser hook, that handles this problem, as the standard json.load() just overwrites the previous object in this case

In [11]:
import json
import pandas as pd
import numpy as np
import os
from datetime import datetime
from pathlib import Path

In [12]:
def as_list_on_duplicate_keys(ordered_pairs):
    """
    A custom JSON object_pairs_hook that collects values for duplicate
    keys into a list. Works well for top-level keys like 'quic'.
    """
    d = {}
    for k, v in ordered_pairs:
        if k in d:
            if isinstance(d[k], list):
                d[k].append(v)
            else:
                d[k] = [d[k], v]
        else:
            d[k] = v
    return d

In [24]:
import os
import pandas as pd
def analyze_quic_capture(json_file_path, encoding='utf-8'):
    """
    Analyzes a decrypted QUIC packet capture from a Wireshark JSON export
    and extracts a set of high-level features for connection migration analysis.

    Args:
        json_file_path (str): The path to the JSON file.

    Returns:
        dict: A dictionary containing the extracted features for the flow.
              Returns None if the capture is empty or invalid.
    """
    with open(json_file_path, 'r', encoding=encoding) as f:
        packets = json.load(f, object_pairs_hook=as_list_on_duplicate_keys)

    if not packets:
        print("Capture file is empty.")
        return None
    
    base_name = os.path.basename(json_file_path)
    file_id = os.path.splitext(base_name)[0]
    print(f"Analyzing file: {base_name}")

    print(file_id.split('_')[0])

    try:
        capture_number = int(file_id.split('_')[0])
    except (ValueError, IndexError):
        capture_number = None

    features = {}
    features['file_id'] = capture_number
    if 'aioquic' in base_name:
        features['implementation'] = 'aioquic'
    elif 'quiche' in base_name:
        features['implementation'] = 'quiche'
    elif 'quicgo' in base_name:
        features['implementation'] = 'quicgo'
    else:
        features['implementation'] = 'unknown'
        
    # --- 1. Initialization and Initial Packet Analysis ---
    first_packet = packets[0]['_source']['layers']
    last_packet = packets[-1]['_source']['layers']

    initial_ip_client = first_packet['ip']['ip.src']
    initial_ip_server = first_packet['ip']['ip.dst']
    initial_port_client = int(first_packet['udp']['udp.srcport'])
    initial_port_server = int(first_packet['udp']['udp.dstport'])
    
    time_first_epoch = float(first_packet['frame']['frame.time_epoch'])
    time_last_epoch = float(last_packet['frame']['frame.time_epoch'])

    features['initial_ip_client'] = initial_ip_client
    features['initial_ip_server'] = initial_ip_server
    features['initial_port_client'] = initial_port_client
    features['initial_port_server'] = initial_port_server
    features['time_first'] = datetime.fromtimestamp(time_first_epoch).isoformat(sep='T', timespec='microseconds').replace(":", "-")
    features['time_last'] = datetime.fromtimestamp(time_last_epoch).isoformat(sep='T', timespec='microseconds').replace(":", "-")
    features['connection_duration'] = (time_last_epoch - time_first_epoch) * 1000  # in ms

    # Initialize counters and flags
    bytes_sent_client = 0
    bytes_sent_server = 0
    packets_sent_client = 0
    packets_sent_server = 0
    quic_packets_sent_server = 0
    quic_packets_sent_client = 0


    # Stream-related counters: number of streams opened
    total_bidi_streams_client_init = 0
    total_bidi_streams_server_init = 0
    total_udi_streams_client_init = 0
    total_udi_streams_server_init = 0

    udi_streams = set()
    bidi_streams = set()
    bytes_bidi_streams_client_init_client_sent = 0
    bytes_bidi_streams_client_init_server_sent = 0
    bytes_bidi_streams_server_init_client_sent = 0
    bytes_bidi_streams_server_init_server_sent = 0
    bytes_udi_streams_client_init = 0
    bytes_udi_streams_server_init = 0

    total_client_app_bytes = 0
    total_server_app_bytes = 0
    client_app_data_chunks = []
    server_app_data_chunks = []

    padding_bytes_in_validation_pc = 0
    padding_bytes_in_validation_pr = 0
    mtu = 0

    # Specific QUIC packet counters
    ack_sent_client = 0
    ack_sent_server = 0
    crypto_sent_client = 0
    crypto_sent_server = 0
    handshake_done_client = 0
    handshake_done_server = 0
    path_challenge_sent_client = 0
    path_challenge_sent_server = 0
    path_response_sent_client = 0
    path_response_sent_server = 0

    features['version_negotiation_occurred'] = 0
    features['retry_occurred'] = 0
    features['new_connection_ids_issued_server'] = 0
    features['retired_cid_count_client'] = 0
    features['retired_cid_count_server'] = 0
    features['new_connection_ids_issued_server'] = 0
    features['new_connection_ids_issued_client'] = 0
    
    # State variables
    migrated = False
    migrated_ip_client = None
    migrated_port_client = None
    time_to_migration = None
    packets_before_migration = 0

    get_request_time = None
    server_data_bytes_after_migration = 0
    last_stream_time = None
    
    handshake_start_time = None
    handshake_end_time = None # We will find the timestamp of the LAST 'Finished' message
    
    path_challenge_data = None
    path_challenge_time = None
    migration_start_time = None
    last_pr_time = None
    
    app_data_before_migration = 0


    # --- 2. Iterate Through All Packets ---
    for i, pkt_data in enumerate(packets):
        layers = pkt_data['_source']['layers']
        
        # Basic packet info
        current_time = float(layers['frame']['frame.time_epoch'])
        packet_len = int(layers['frame']['frame.len'])
        
        src_ip = layers['ip']['ip.src']
        dst_ip = layers['ip']['ip.dst']

        src_port = int(layers['udp']['udp.srcport'])
        dst_port = int(layers['udp']['udp.dstport'])
        
        if not migrated and \
           (dst_ip == initial_ip_server and dst_port == initial_port_server) and \
           (src_ip != initial_ip_client or src_port != initial_port_client):
            
            migrated = True
            migration_start_time = current_time
            print(f"Migration detected at packet {i}, time {current_time}, packet len {packet_len}")

            migrated_ip_client = src_ip
            migrated_port_client = src_port
            
            time_to_migration = (current_time - time_first_epoch) * 1000
            packets_before_migration = i
            
            ip_changed = src_ip != initial_ip_client
            port_changed = src_port != initial_port_client
            if ip_changed and port_changed:
                features['migration_type'] = 'IP_AND_PORT'
            elif ip_changed:
                features['migration_type'] = 'IP_ONLY'
            elif port_changed:
                features['migration_type'] = 'PORT_ONLY'


        is_client_pkt = (src_ip == initial_ip_client or src_ip == migrated_ip_client)

        # Update byte and packet counters
        if is_client_pkt:
            bytes_sent_client += packet_len
            packets_sent_client += 1
        else:
            bytes_sent_server += packet_len
            packets_sent_server += 1
        quic_packet_list = layers['quic']
        http_packet_list = layers['http3'] if 'http3' in layers else []

        if not isinstance(quic_packet_list, list):
            quic_packet_list = [quic_packet_list]
        

        # 2. Loop through each QUIC packet within the UDP datagram.
        for quic_packet in quic_packet_list:
            if quic_packet.get('quic.version') == '0x00000000':
                features['version_negotiation_occurred'] = 1
            if quic_packet.get('quic.long.packet_type') == '3': # Retry packet
                features['retry_occurred'] = 1
            if quic_packet.get('quic.long.packet_type') == '0': # Initial packet
                if handshake_start_time is None:
                    handshake_start_time = current_time

            # Analyze QUIC frames if they exist
            # 3. Get frames from the current quic_packet, not from layers['quic'].
            quic_frames = quic_packet.get('quic.frame', [])
            if not isinstance(quic_frames, list): # If there's only one frame, it's a dict
                quic_frames = [quic_frames]
            
            # These flags help identify which packet contains padding
            is_path_challenge_pkt = any(f.get('quic.frame_type') == '0x000000000000001a' for f in quic_frames)
            is_path_response_pkt = any(f.get('quic.frame_type') == '0x000000000000001b' for f in quic_frames)
                
            for frame in quic_frames:
                frame_type = frame.get('quic.frame_type')
                if not frame_type: continue # Skip if frame is empty

                if is_client_pkt:
                    quic_packets_sent_client += 1
                else:
                    quic_packets_sent_server += 1
                
                if frame_type == '0x0000000000000006': # CRYPTO frame
                    if is_client_pkt:
                        crypto_sent_client += 1
                    else:
                        crypto_sent_server += 1
                
                if frame_type in ('0x0000000000000002', '0x0000000000000003'): # ACK frame
                    if is_client_pkt:
                        ack_sent_client += 1
                    else:
                        ack_sent_server += 1
                
                if frame_type == '0x000000000000001e': # HANDSHAKE_DONE frame
                    if handshake_end_time is None:
                        handshake_end_time = current_time
                    if is_client_pkt:
                        handshake_done_client += 1
                    else:
                        handshake_done_server += 1
                
                if frame_type == '0x0000000000000018': # NEW_CONNECTION_ID
                    if not is_client_pkt:
                        features['new_connection_ids_issued_server'] += 1
                    else:
                        # Note: clients do not issue CIDs, but keeping your logic
                        features['new_connection_ids_issued_client'] += 1
                
                if frame_type == '0x0000000000000019': # RETIRE_CONNECTION_ID
                    if is_client_pkt:
                        features['retired_cid_count_client'] += 1
                    else:
                        features['retired_cid_count_server'] += 1

                if frame_type == '0x000000000000001a': # PATH_CHALLENGE
                    if is_client_pkt:
                        features['path_validation_initiated'] = 1
                        path_challenge_data = frame['quic.path_challenge.data']
                        path_challenge_time = current_time
                        mtu = max(mtu, packet_len)
                    if is_client_pkt:
                        path_challenge_sent_client += 1
                    else:
                        path_challenge_sent_server += 1

                if frame_type == '0x000000000000001b': # PATH_RESPONSE
                    if not is_client_pkt:
                        if path_challenge_data and frame['quic.path_response.data'] == path_challenge_data:
                            features['first_path_validation_response_latency'] = (current_time - path_challenge_time) * 1000
                            #features['migration_duration'] = (current_time - migration_start_time) * 1000
                    last_pr_time = current_time
                    print(f"PATH_RESPONSE at time {current_time}, packet len {packet_len}")
                    if is_client_pkt:
                        path_response_sent_client += 1
                    else:
                        path_response_sent_server += 1
            
                # Padding is its own frame type, not a field on other frames.
                if frame_type == '0x0000000000000000': # PADDING frame
                    padding_len = int(frame.get('quic.padding_length', 1)) # Padding can be 1 byte if length is omitted
                    if is_path_challenge_pkt:
                        padding_bytes_in_validation_pc = padding_len
                    if is_path_response_pkt:
                        padding_bytes_in_validation_pr = padding_len
                
                if frame_type in ('0x0000000000000008', '0x0000000000000009', '0x000000000000000a', '0x000000000000000b', '0x000000000000000c', '0x000000000000000d', '0x000000000000000e', '0x000000000000000f'): # STREAM frames
                    initiator = int(frame.get('quic.stream.stream_id_tree').get('quic.stream.initiator'),0)
                    direction = int(frame.get('quic.stream.stream_id_tree').get('quic.stream.direction'),0)
                    try:
                        data_len = frame.get('quic.stream.length')
                        if data_len:
                            bytes = int(data_len)
                        else:
                            stream_data = frame.get('quic.stream_data')
                            if stream_data:
                                bytes = len(stream_data.replace(':', '')) // 2  # Hex string to bytes
                    except (ValueError, TypeError):
                        bytes = 0

                    id = int(frame.get('quic.stream.stream_id', 0))
                    last_stream_time = current_time if is_client_pkt == False else last_stream_time
                    if not migrated:
                        app_data_before_migration += bytes

                    if is_client_pkt:
                        total_client_app_bytes += bytes
                        client_app_data_chunks.append(bytes)
                    else:
                        total_server_app_bytes += bytes
                        server_app_data_chunks.append(bytes)
                    
                    if direction == 0:
                        if initiator == 0:
                            if id not in bidi_streams:
                                total_bidi_streams_client_init += 1
                                bidi_streams.add(id)
                            if is_client_pkt:
                                
                                bytes_bidi_streams_client_init_client_sent += bytes
                            else:
                                bytes_bidi_streams_client_init_server_sent += bytes
                                
                        else:
                            if id not in bidi_streams:
                                total_bidi_streams_server_init += 1
                                bidi_streams.add(id)
                            if is_client_pkt:
                                bytes_bidi_streams_server_init_client_sent += bytes
                            else:
                                bytes_bidi_streams_server_init_server_sent += bytes
                    else:
                        if initiator == 0:
                            if id not in udi_streams:
                                total_udi_streams_client_init += 1
                            bytes_udi_streams_client_init += bytes
                        else:
                            if id not in udi_streams:
                                total_udi_streams_server_init += 1
                            bytes_udi_streams_server_init += bytes

                if frame_type in ('0x000000000000001c', '0x000000000000001d'): # CONNECTION_CLOSE
                    features['connection_close_type'] = 'CLIENT_CLOSE' if is_client_pkt else 'SERVER_CLOSE'

        if is_client_pkt and get_request_time is None and 'http3' in layers:
            http_packet_list = layers['http3']
            if not isinstance(http_packet_list, list):
                http_packet_list = [http_packet_list]

            for http_packet in http_packet_list:
                if 'http3.stream' in http_packet:
                    streams = http_packet.get('http3.stream',[])
                    
                    if not isinstance(streams, list):
                        streams = [streams]
                    for st in streams:
                        frames = st.get('http3.frame', [])
                        if not isinstance(frames, list):
                            frames = [frames] 
                        
                        for frame in frames:

                            #if frame.get('http3.frame_type') != '0x0000000000000001': continue
                            headers = frame.get('http3.headers.header', [])
                            if not isinstance(headers, list):
                                headers = [headers] 
                                 
                            for header in headers:
                                if header.get('http3.header.header.name') == ':method' and header.get('http3.headers.header.value') == 'GET':
                                    get_request_time = current_time
                                    break
                            if get_request_time: break
                        if get_request_time: break
                    if get_request_time: break
                                
            


    # --- 3. Final Calculations and Assembly ---
    features['bytes_sent_client'] = bytes_sent_client
    features['bytes_sent_server'] = bytes_sent_server
    features['packets_sent_client'] = packets_sent_client
    features['packets_sent_server'] = packets_sent_server
    features['quic_packets_sent_client'] = quic_packets_sent_client
    features['quic_packets_sent_server'] = quic_packets_sent_server
    
    if handshake_end_time:
        features['handshake_duration'] = (handshake_end_time - handshake_start_time) * 1000
    else:
        features['handshake_duration'] = None # Handshake did not complete or was not found
        
    features['time_to_migration'] = time_to_migration
    features['migration_duration'] = (last_pr_time - migration_start_time) * 1000
    print(f'migration_start: {migration_start_time}, last_pr_time: {last_pr_time}')

    features['packets_before_migration'] = packets_before_migration
    features['app_data_bytes_before_migration'] = app_data_before_migration


    if not migrated:
        features['migration_type'] = 'NO_MIGRATION'
        features['time_to_migration'] = 0
        features['packets_before_migration'] = 0
        features['migration_duration'] = 0
        features['path_validation_initiated'] = 0
        features['first_' \
        'first_path_validation_response_latency'] = 0
    elif migration_start_time < get_request_time:
        features['migration_type'] = 'BEFORE_DOWNLOAD'
    elif  migration_start_time < last_stream_time:
        features['migration_type'] = 'DURING_DOWNLOAD'
    elif migration_start_time > last_stream_time:
        features['migration_type'] = 'AFTER_DOWNLOAD'
    else:
        features['migration_type'] = 'UNKNOWN'
    
    # Set migration-related features to 0 or null if no migration occurred
    if not migrated:
        features['migration_type'] = 'NO_MIGRATION'
        features['time_to_migration'] = 0
        features['packets_before_migration'] = 0
        features['migration_duration'] = 0
        features['path_validation_initiated'] = 0
        features['first_' \
        'first_path_validation_response_latency'] = 0

    # These features couldn't be accurately determined from this specific JSON structure but are included as placeholders
    features['padding_bytes_in_validation_pc'] = padding_bytes_in_validation_pc
    features['padding_bytes_in_validation_pr'] = padding_bytes_in_validation_pr
    features['mtu'] = mtu
    
    features['total_bidi_streams_client_init'] = total_bidi_streams_client_init
    features['total_bidi_streams_server_init'] = total_bidi_streams_server_init
    features['total_udi_streams_client_init'] = total_udi_streams_client_init
    features['total_udi_streams_server_init'] = total_udi_streams_server_init
    features['bytes_bidi_streams_client_init_client_sent'] = bytes_bidi_streams_client_init_client_sent
    features['bytes_bidi_streams_client_init_server_sent'] = bytes_bidi_streams_client_init_server_sent
    features['bytes_bidi_streams_server_init_client_sent'] = bytes_bidi_streams_server_init_client_sent
    features['bytes_bidi_streams_server_init_server_sent'] = bytes_bidi_streams_server_init_server_sent
    features['bytes_udi_streams_client_init'] = bytes_udi_streams_client_init
    features['bytes_udi_streams_server_init'] = bytes_udi_streams_server_init

    features['total_client_app_bytes'] = total_client_app_bytes
    features['total_server_app_bytes'] = total_server_app_bytes
    features['avg_request_size'] = np.mean(client_app_data_chunks) if client_app_data_chunks else 0
    features['avg_response_size'] = np.mean(server_app_data_chunks) if server_app_data_chunks else 0

    features['ack_sent_client'] = ack_sent_client
    features['ack_sent_server'] = ack_sent_server
    features['crypto_sent_client'] = crypto_sent_client
    features['crypto_sent_server'] = crypto_sent_server
    features['handshake_done_client'] = handshake_done_client
    features['handshake_done_server'] = handshake_done_server
    features['path_challenge_sent_client'] = path_challenge_sent_client
    features['path_challenge_sent_server'] = path_challenge_sent_server
    features['path_response_sent_client'] = path_response_sent_client
    features['path_response_sent_server'] = path_response_sent_server
    features['app_data_bytes_before_migration'] = app_data_before_migration

    


    # Final check for connection close type
    if 'connection_close_type' not in features:
        features['connection_close_type'] = 'IDLE_TIMEOUT' # Default assumption
        
    return features

In [25]:
import os

def process_quic_capture(json_file_path):
    """
    Determines the correct encoding (UTF-8 or UTF-16) for a JSON file
    and then calls the analysis function.
    """

    if not os.path.exists(json_file_path) or os.path.getsize(json_file_path) == 0:
        print(f"Skipping '{json_file_path}': File is empty or does not exist.\n")
        return

    try:
        extracted_features = analyze_quic_capture(json_file_path, encoding='utf-8')
        return extracted_features

    except UnicodeDecodeError:
        try:
            extracted_features = analyze_quic_capture(json_file_path, encoding='utf-16')
            return extracted_features
        except (json.JSONDecodeError, UnicodeDecodeError) as e:
            print(f"    -> ERROR: Failed to process '{os.path.basename(json_file_path)}' as UTF-16. Error: {e}")

    except json.JSONDecodeError as e:

        print(f"    -> ERROR: File is UTF-8 but has invalid JSON. Error: {e}")

In [26]:
pd.set_option('display.max_columns', None)

In [27]:
json_file = os.path.join("..","captures", "captures_json", "quicgo", "5922_quicgo.json")
extracted_features = process_quic_capture(json_file)

Analyzing file: 5922_quicgo.json
5922
Migration detected at packet 9, time 1764097151.153092, packet len 1232
PATH_RESPONSE at time 1764097151.153149, packet len 1232
PATH_RESPONSE at time 1764097151.153198, packet len 72
migration_start: 1764097151.153092, last_pr_time: 1764097151.153198


In [28]:
if extracted_features:
    df = pd.DataFrame([extracted_features], index=[extracted_features['file_id']])
    print("Extracted Features (Corrected Migration & Direction Logic):")
    display(df)

Extracted Features (Corrected Migration & Direction Logic):


,file_id,implementation,initial_ip_client,initial_ip_server,initial_port_client,initial_port_server,time_first,time_last,connection_duration,version_negotiation_occurred,retry_occurred,new_connection_ids_issued_server,retired_cid_count_client,retired_cid_count_server,new_connection_ids_issued_client,migration_type,path_validation_initiated,first_path_validation_response_latency,connection_close_type,bytes_sent_client,bytes_sent_server,packets_sent_client,packets_sent_server,quic_packets_sent_client,quic_packets_sent_server,handshake_duration,time_to_migration,migration_duration,packets_before_migration,app_data_bytes_before_migration,padding_bytes_in_validation_pc,padding_bytes_in_validation_pr,mtu,total_bidi_streams_client_init,total_bidi_streams_server_init,total_udi_streams_client_init,total_udi_streams_server_init,bytes_bidi_streams_client_init_client_sent,bytes_bidi_streams_client_init_server_sent,bytes_bidi_streams_server_init_client_sent,bytes_bidi_streams_server_init_server_sent,bytes_udi_streams_client_init,bytes_udi_streams_server_init,total_client_app_bytes,total_server_app_bytes,avg_request_size,avg_response_size,ack_sent_client,ack_sent_server,crypto_sent_client,crypto_sent_server,handshake_done_client,handshake_done_server,path_challenge_sent_client,path_challenge_sent_server,path_response_sent_client,path_response_sent_server
5922,5922,quicgo,127.0.0.2,127.0.0.1,57405,6121,2025-11-25T19-59-11.151136,2025-11-25T19-59-11.153676,2.540112,0,0,3,1,1,3,BEFORE_DOWNLOAD,1,0.056982,CLIENT_CLOSE,5676,4612,9,8,21,20,1.880169,1.955986,0.106096,9,10,1159,1159,1232,1,0,1,1,39,126,0,0,8,10,47,136,23.5,68.0,4,5,5,3,0,1,1,1,1,1


#### Loading one json for test

In [ ]:
json_file = '..\captures_json\quiche\quiche_capture_20.json'
import os
if os.path.getsize(json_file) > 0:
    with open(json_file, 'r', encoding='utf-16') as f:
        try:
            extracted_features = analyze_quic_capture(json_file, encoding='utf-16')
        except json.JSONDecodeError as e:
            print(f"Error decoding JSON: {e}")
else:
    print("The JSON file is empty.")

In [ ]:
if extracted_features:
    df = pd.DataFrame([extracted_features])#, index=[extracted_features['ID']])
    print("Extracted Features (Corrected Migration & Direction Logic):")
    display(df)

Extracted Features (Corrected Migration & Direction Logic):


,file_id,implementation,initial_ip_client,initial_ip_server,initial_port_client,initial_port_server,time_first,time_last,connection_duration,version_negotiation_occurred,retry_occurred,new_connection_ids_issued_server,retired_cid_count_client,retired_cid_count_server,new_connection_ids_issued_client,migration_type,path_validation_initiated,first_path_validation_response_latency,connection_close_type,bytes_sent_client,bytes_sent_server,packets_sent_client,packets_sent_server,quic_packets_sent_client,quic_packets_sent_server,handshake_duration,time_to_migration,migration_duration,packets_before_migration,app_data_bytes_before_migration,padding_bytes_in_validation_pc,padding_bytes_in_validation_pr,mtu,total_bidi_streams_client_init,total_bidi_streams_server_init,total_udi_streams_client_init,total_udi_streams_server_init,bytes_bidi_streams_client_init_client_sent,bytes_bidi_streams_client_init_server_sent,bytes_bidi_streams_server_init_client_sent,bytes_bidi_streams_server_init_server_sent,bytes_udi_streams_client_init,bytes_udi_streams_server_init,ack_sent_client,ack_sent_server,crypto_sent_client,crypto_sent_server,handshake_done_client,handshake_done_server,path_challenge_sent_client,path_challenge_sent_server,path_response_sent_client,path_response_sent_server
0,20,quiche,127.0.0.2,127.0.0.1,54110,4433,2025-10-20T18-32-53.659294,2025-10-20T18-32-53.678830,19.536018,1,1,1,0,0,1,BEFORE_DOWNLOAD,1,1.641989,CLIENT_CLOSE,7269,4649,14,12,19,19,11.227846,12.987137,2.469063,13,47,1294,1294,1382,1,0,4,4,72,308,0,0,47,47,5,5,3,4,0,1,1,1,1,1


#### Load all jsons from a directory

In [34]:
directory_path_quiche = r'C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\test_gan_data\captures_json\quiche'
directory_path_aioquic = r'C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures_json\aioquic\aioquic'
git_directory_path = r'C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\test'
all_extracted_features = []

In [35]:
def list_files_recursively(directory, extension=".json", files=[]):
    files = []
    for entry in os.listdir(directory):
        full_path = os.path.join(directory, entry)

        if os.path.isdir(full_path):
            files.extend(list_files_recursively(full_path, extension))
        elif entry.endswith(extension):
            files.append(full_path)

    return files

In [36]:
json_files = list_files_recursively(git_directory_path) 

In [37]:
json_files

['C:\\Users\\vassa\\Desktop\\UZH\\Masters Project\\synthetic_network_data_gen\\captures\\test\\1_quiche_capture.json',
 'C:\\Users\\vassa\\Desktop\\UZH\\Masters Project\\synthetic_network_data_gen\\captures\\test\\2_quiche_capture.json',
 'C:\\Users\\vassa\\Desktop\\UZH\\Masters Project\\synthetic_network_data_gen\\captures\\test\\3172_aioquic_before_fast.json',
 'C:\\Users\\vassa\\Desktop\\UZH\\Masters Project\\synthetic_network_data_gen\\captures\\test\\3173_aioquic_before_fast.json',
 'C:\\Users\\vassa\\Desktop\\UZH\\Masters Project\\synthetic_network_data_gen\\captures\\test\\3174_aioquic_before_fast.json',
 'C:\\Users\\vassa\\Desktop\\UZH\\Masters Project\\synthetic_network_data_gen\\captures\\test\\3175_aioquic_before_fast.json',
 'C:\\Users\\vassa\\Desktop\\UZH\\Masters Project\\synthetic_network_data_gen\\captures\\test\\3941_aioquic_before_slow_600.json',
 'C:\\Users\\vassa\\Desktop\\UZH\\Masters Project\\synthetic_network_data_gen\\captures\\test\\3942_aioquic_before_slow_600

In [38]:
len(json_files)

29

In [39]:
for path in json_files:
    name = os.path.basename(path)

    print(f"Processing '{name}'...")
    
    # Process the file and get the features dictionary
    features = process_quic_capture(path)
    
    # If features were successfully extracted, add them to our list
    if features:
        all_extracted_features.append(features)
        print(f"-> Successfully extracted features from '{name}'.\n")


Processing '1_quiche_capture.json'...
Analyzing file: 1_quiche_capture.json
1
Migration detected at packet 13, time 1762181365.11753, packet len 1382
PATH_RESPONSE at time 1762181365.118052, packet len 1382
PATH_RESPONSE at time 1762181365.118393, packet len 84
migration_start: 1762181365.11753, last_pr_time: 1762181365.118393
-> Successfully extracted features from '1_quiche_capture.json'.

Processing '2_quiche_capture.json'...
Analyzing file: 2_quiche_capture.json
2
Migration detected at packet 13, time 1762181367.770805, packet len 1382
PATH_RESPONSE at time 1762181367.771365, packet len 1382
PATH_RESPONSE at time 1762181367.771945, packet len 84
migration_start: 1762181367.770805, last_pr_time: 1762181367.771945
-> Successfully extracted features from '2_quiche_capture.json'.

Processing '3172_aioquic_before_fast.json'...
Analyzing file: 3172_aioquic_before_fast.json
3172
Migration detected at packet 7, time 1763398057.039803, packet len 74
PATH_RESPONSE at time 1763398057.05521, p

In [40]:
print("------------------------------------------")
print("Finished processing all files.")

if all_extracted_features:
    df = pd.DataFrame(all_extracted_features)
    
    # Optional: Set one of the columns as the index
    # df.set_index('ID', inplace=True)

    print("Final DataFrame with all extracted features:")
    display(df) # Use display() if in a Jupyter Notebook, otherwise use print(df)
else:
    print("No features were extracted. The final DataFrame is empty.")

------------------------------------------
Finished processing all files.
Final DataFrame with all extracted features:


,file_id,implementation,initial_ip_client,initial_ip_server,initial_port_client,initial_port_server,time_first,time_last,connection_duration,version_negotiation_occurred,retry_occurred,new_connection_ids_issued_server,retired_cid_count_client,retired_cid_count_server,new_connection_ids_issued_client,migration_type,path_validation_initiated,first_path_validation_response_latency,connection_close_type,bytes_sent_client,bytes_sent_server,packets_sent_client,packets_sent_server,quic_packets_sent_client,quic_packets_sent_server,handshake_duration,time_to_migration,migration_duration,packets_before_migration,app_data_bytes_before_migration,padding_bytes_in_validation_pc,padding_bytes_in_validation_pr,mtu,total_bidi_streams_client_init,total_bidi_streams_server_init,total_udi_streams_client_init,total_udi_streams_server_init,bytes_bidi_streams_client_init_client_sent,bytes_bidi_streams_client_init_server_sent,bytes_bidi_streams_server_init_client_sent,bytes_bidi_streams_server_init_server_sent,bytes_udi_streams_client_init,bytes_udi_streams_server_init,total_client_app_bytes,total_server_app_bytes,avg_request_size,avg_response_size,ack_sent_client,ack_sent_server,crypto_sent_client,crypto_sent_server,handshake_done_client,handshake_done_server,path_challenge_sent_client,path_challenge_sent_server,path_response_sent_client,path_response_sent_server
0,1,quiche,127.0.0.2,127.0.0.1,62159,4433,2025-11-03T15-49-25.106530,2025-11-03T15-49-25.149371,42.840958,1,1,1,0,0,1,BEFORE_DOWNLOAD,1.0,0.521898,CLIENT_CLOSE,7269,4568,14,12,19,19,9.505033,11.000156,0.862837,13,47,1294,1294,1382,1,0,4,4,72,230,0,0,47,47,119,277,23.8,55.400000,5,5,3,4,0,1,1,1,1,1
1,2,quiche,127.0.0.2,127.0.0.1,52040,4433,2025-11-03T15-49-27.761369,2025-11-03T15-49-27.775134,13.765097,1,1,1,0,0,1,BEFORE_DOWNLOAD,1.0,0.560045,CLIENT_CLOSE,7194,4568,13,12,18,19,7.610083,9.435892,1.140118,13,47,1294,1294,1382,1,0,4,4,72,230,0,0,47,47,119,277,23.8,55.400000,4,5,3,4,0,1,1,1,1,1
2,3172,aioquic,127.0.0.2,127.0.0.1,49668,4433,2025-11-17T17-47-37.011112,2025-11-17T17-47-37.060358,49.246073,0,0,7,0,0,7,BEFORE_DOWNLOAD,NaN,NaN,CLIENT_CLOSE,3153,3090,7,7,22,19,16.601086,28.691053,15.407085,7,34,0,0,0,1,0,4,3,28,878,0,0,20,17,48,895,9.6,223.750000,5,3,2,3,0,1,0,1,1,0
3,3173,aioquic,127.0.0.2,127.0.0.1,50461,4433,2025-11-17T17-47-40.410962,2025-11-17T17-47-40.433849,22.886992,0,0,7,0,0,7,BEFORE_DOWNLOAD,NaN,NaN,CLIENT_CLOSE,3207,3027,8,6,22,18,9.938002,9.589911,10.220051,5,17,0,0,0,1,0,4,3,28,878,0,0,20,17,48,895,9.6,223.750000,4,2,2,3,0,1,0,1,1,0
4,3174,aioquic,127.0.0.2,127.0.0.1,49669,4433,2025-11-17T17-47-43.799772,2025-11-17T17-47-43.824725,24.952888,0,0,7,0,0,7,BEFORE_DOWNLOAD,NaN,NaN,CLIENT_CLOSE,3207,3027,8,6,22,18,10.686874,9.984970,11.665106,5,17,0,0,0,1,0,4,3,28,878,0,0,20,17,48,895,9.6,223.750000,4,2,2,3,0,1,0,1,1,0
5,3175,aioquic,127.0.0.2,127.0.0.1,55759,4433,2025-11-17T17-47-47.151185,2025-11-17T17-47-47.182836,31.651020,0,0,7,0,0,7,BEFORE_DOWNLOAD,NaN,NaN,CLIENT_CLOSE,3094,2966,6,5,22,18,11.970997,12.630939,15.383959,6,34,0,0,0,1,0,4,3,28,878,0,0,20,17,48,895,9.6,223.750000,5,2,2,3,0,1,0,1,1,0
6,3941,aioquic,127.0.0.2,127.0.0.1,49871,4433,2025-11-17T18-44-04.289818,2025-11-17T18-44-04.506589,216.770887,0,0,7,0,0,7,BEFORE_DOWNLOAD,NaN,NaN,CLIENT_CLOSE,3218,3289,8,10,23,24,10.715961,14.279842,30.022144,7,34,0,0,0,1,0,4,3,28,881,0,0,20,17,48,898,9.6,149.666667,6,4,2,3,0,1,0,1,1,0
7,3942,aioquic,127.0.0.2,127.0.0.1,54659,4433,2025-11-17T18-44-08.700439,2025-11-17T18-44-08.894267,193.828106,0,0,7,0,0,7,BEFORE_DOWNLOAD,NaN,NaN,CLIENT_CLOSE,3344,3228,9,9,24,22,10.530949,10.316133,22.801876,5,17,0,0,0,1,0,4,3,28,881,0,0,20,17,48,898,9.6,149.666667,5,4,3,3,0,1,0,1,1,0
8,3943,aioquic,127.0.0.2,127.0.0.1,54661,4433,2025-11-17T18-44-13.145398,2025-11-17T18-44-13.334483,189.085007,0,0,7,0,0,7,BEFORE_DOWNLOAD,NaN,NaN,CLIENT_CLOSE,3337,3165,10,8,24,21,10.019064,9.534121,20.253897,5,17,0,0,0,1,0,4,3,28,881,0,0,20,17,48,898,9.6,149.666667,6,3,2,3,0,1,0,1,1,0
9,3944,aioquic,127.0.0.2,127.0

In [42]:
df.to_csv(r'C:\Users\vassa\Desktop\UZH\Masters Project\synthetic_network_data_gen\captures\test\high_level\data.csv', index=False)